# Train deepfake-detect on Kaggle

Trains the linear head on **FaceForensics++ (c23)** using the official train/val/test splits, then tests on **Celeb-DF v2**, a dataset the model never sees during training. That second number is the honest one.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (or P100). Turn **Internet on**.
2. Both datasets need an access request to their authors ([FaceForensics++](https://github.com/ondyari/FaceForensics), [Celeb-DF](https://github.com/yuezunli/celeb-deepfakeforensics)). Their licences forbid redistribution, so upload your copy as a **private** Kaggle dataset and attach it with *Add Input*.
3. Set the two paths in the next code cell to wherever your upload lands under `/kaggle/input/`.

Expected runtime: ~1–2 h, mostly video decoding and face detection. Embeddings are cached, so re-runs take minutes.

In [1]:
REPO = "https://github.com/CodeRafay/deepfake-detect"
BRANCH = "redesign"
!rm -rf /kaggle/working/deepfake-detect
!git clone -q -b $BRANCH $REPO /kaggle/working/deepfake-detect
!pip install -q /kaggle/working/deepfake-detect pytest
!python -m pytest -q /kaggle/working/deepfake-detect/tests


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
...                                                                      [100%]
3 passed in 5.68s


In [2]:
import json, urllib.request
from pathlib import Path


FFPP = Path("/kaggle/input/notebooks/rafayadeel/ffpp-c23-download/ffpp")
CELEBDF = Path("/kaggle/input/datasets/rafayadeel/celeb-df-v2")

# FFPP = Path("/kaggle/input/YOUR-FFPP-UPLOAD")        # contains original_sequences/ and manipulated_sequences/
# CELEBDF = Path("/kaggle/input/YOUR-CELEBDF-UPLOAD")  # contains Celeb-real/, Celeb-synthesis/, YouTube-real/, List_of_testing_videos.txt
METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
DATA, CDF = Path("/kaggle/working/data"), Path("/kaggle/working/celebdf")

def link(src, dst):
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(src)

# Official FF++ splits are by source-video pair, so no identity appears in two splits.
for split in ["train", "val", "test"]:
    url = f"https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{split}.json"
    for a, b in json.load(urllib.request.urlopen(url)):
        for v in (a, b):
            link(FFPP / f"original_sequences/youtube/c23/videos/{v}.mp4", DATA / split / "real" / f"{v}.mp4")
        for m in METHODS:
            for x, y in ((a, b), (b, a)):
                link(FFPP / f"manipulated_sequences/{m}/c23/videos/{x}_{y}.mp4", DATA / split / "fake" / f"{m}_{x}_{y}.mp4")

# Celeb-DF official test list: "1 path" = real, "0 path" = fake.
for line in (CELEBDF / "List_of_testing_videos.txt").read_text().splitlines():
    if line.strip():
        lab, rel = line.split()
        link(CELEBDF / rel, CDF / ("real" if lab == "1" else "fake") / rel.replace("/", "_"))

for d in [DATA / s / c for s in ["train", "val", "test"] for c in ["real", "fake"]] + [CDF / "real", CDF / "fake"]:
    print(f"{d}: {len(list(d.glob('*')))} videos")

/kaggle/working/data/train/real: 720 videos
/kaggle/working/data/train/fake: 2880 videos
/kaggle/working/data/val/real: 140 videos
/kaggle/working/data/val/fake: 560 videos
/kaggle/working/data/test/real: 140 videos
/kaggle/working/data/test/fake: 560 videos
/kaggle/working/celebdf/real: 178 videos
/kaggle/working/celebdf/fake: 340 videos


In [3]:
!cd /kaggle/working && python -m deepfake_detect.train --data data --eval celebdf=celebdf --out head.safetensors

preprocessor_config.json: 100%|████████████████| 316/316 [00:00<00:00, 1.41MB/s]
config.json: 100%|█████████████████████████| 4.52k/4.52k [00:00<00:00, 9.17MB/s]

model.safetensors: downloading bytes: ████▏                 |  323MB, 26.4MB/s  
model.safetensors: downloading bytes: ████▉                 |  386MB, 33.3MB/s  
model.safetensors: downloading bytes: █████████████▋        | 1.06GB, 88.3MB/s  
model.safetensors: downloading bytes: ██████████████▎       | 1.11GB, 90.4MB/s  
model.safetensors: downloading bytes: ███████████████       | 1.17GB, 90.8MB/s  
model.safetensors: downloading bytes: ██████████████████████| 1.18GB, 88.6MB/s  
model.safetensors: reconstructing file: 100%|██████| 1.71GB / 1.71GB,  137MB/s  
Loading weights: 100%|██████████████████████| 392/392 [00:00<00:00, 1061.09it/s]
[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
----------------

Paste the table above into the README's **Results** section. Report it as is, even if the Celeb-DF number is unflattering. Next, sanity-check a few test videos:

In [4]:
from deepfake_detect import Detector
det = Detector("/kaggle/working/head.safetensors")
for p in sorted((CDF / "real").glob("*"))[:3] + sorted((CDF / "fake").glob("*"))[:3]:
    r = det.predict(p)
    print(p.parent.name, r["label"], r["score"])

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{

real real 0.2108
real fake 0.6914
real real 0.4204
fake fake 0.8935
fake fake 0.7728
fake fake 0.9848


## Publish the head (optional)
Add a Kaggle secret `HF_TOKEN` (write access), then upload. Users get it automatically with `Detector()` once `DEFAULT_HEAD` in `model.py` points to your repo.

In [5]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo = "CodeRafay/deepfake-detect"
api.create_repo(repo, exist_ok=True)
for f in ["head.safetensors", "head.metrics.json"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/{f}", path_in_repo=f, repo_id=repo)

BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 139036765 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.